# Codificador Semantico
Ejecuta **todas las celdas** (Entorno de ejecucion -> Ejecutar todo) y abre la URL que aparece abajo.

**Requisito:** Cuenta Google para usar Colab.

In [ ]:
# Celda 1: Instalar dependencias
!pip install flask requests pyngrok -q
print('OK Dependencias instaladas')

In [ ]:
# Celda 2: Configuracion
import os, zipfile, shutil, requests as req_dl

# Configuracion hardcodeada OpenRouter + Haiku 4.5
API_KEY = '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'
MODEL   = 'anthropic/claude-haiku-4.5'
API_URL = 'https://openrouter.ai/api/v1/chat/completions'
PORT    = 8765

# Descargar dist/ desde GitHub Release publico
DIST_ZIP = 'https://github.com/aacevedoc-ds/Cod_AA/releases/download/v1.0.0/dist.zip'
DIST_DIR = '/content/dist'

if not os.path.exists(DIST_DIR):
    print('Descargando app...')
    r = req_dl.get(DIST_ZIP, timeout=60)
    r.raise_for_status()
    with open('/tmp/dist.zip', 'wb') as f:
        f.write(r.content)
    with zipfile.ZipFile('/tmp/dist.zip') as z:
        z.extractall('/content')
    print('OK App descargada en ' + DIST_DIR)
else:
    print('OK App ya disponible en ' + DIST_DIR)

In [ ]:
# Celda 3: Servidor Flask (proxy LLM + archivos estaticos)
from flask import Flask, request, jsonify, send_from_directory
import requests as req, threading, os

app = Flask(__name__)
MODEL_NAME = MODEL + ':cloud'

@app.route('/api/config')
def config():
    return jsonify({'api_url': API_URL, 'api_key': API_KEY, 'model': MODEL})

@app.route('/api/tags')
def tags():
    return jsonify({'models': [{'name': MODEL_NAME}]})

@app.route('/api/chat', methods=['POST', 'OPTIONS'])
def chat():
    if request.method == 'OPTIONS':
        resp = app.make_response('')
        resp.headers['Access-Control-Allow-Origin'] = '*'
        resp.headers['Access-Control-Allow-Headers'] = 'Content-Type,Authorization,X-Proxy-Auth'
        resp.headers['Access-Control-Allow-Methods'] = 'POST, OPTIONS'
        return resp
    data = request.json or {}
    messages = data.get('messages', [])
    auth_header = request.headers.get('X-Proxy-Auth') or request.headers.get('Authorization') or ('Bearer ' + API_KEY)
    body = {'model': MODEL, 'messages': messages, 'max_tokens': 32768}
    try:
        r = req.post(API_URL, headers={'Authorization': auth_header, 'Content-Type': 'application/json'}, json=body, timeout=300)
        result = r.json()
        if 'error' in result:
            return jsonify({'error': result['error']}), 500
        content = result.get('choices', [{}])[0].get('message', {}).get('content', '')
        resp = jsonify({'message': {'content': content}})
        resp.headers['Access-Control-Allow-Origin'] = '*'
        return resp
    except Exception as e:
        return jsonify({'error': str(e)}), 502

@app.route('/proxy', methods=['GET', 'POST', 'OPTIONS'])
def proxy():
    if request.method == 'OPTIONS':
        resp = app.make_response('')
        resp.headers['Access-Control-Allow-Origin'] = '*'
        resp.headers['Access-Control-Allow-Headers'] = 'Content-Type,Authorization,X-Proxy-Auth'
        resp.headers['Access-Control-Allow-Methods'] = 'GET, POST, OPTIONS'
        return resp
    target_url = request.args.get('url', '')
    if not target_url:
        return jsonify({'error': 'url param required'}), 400
    auth_header = request.headers.get('X-Proxy-Auth') or request.headers.get('Authorization') or ('Bearer ' + API_KEY)
    headers = {'Authorization': auth_header, 'Content-Type': 'application/json'}
    try:
        if request.method == 'POST':
            body = {**request.json} if request.json else {}
            body['max_tokens'] = 32768
            if 'moonshot.ai' in target_url:
                body.pop('temperature', None)
            r = req.post(target_url, headers=headers, json=body, timeout=300)
        else:
            r = req.get(target_url, headers=headers, timeout=30)
        resp = app.make_response(r.content)
        resp.status_code = r.status_code
        resp.headers['Content-Type'] = r.headers.get('Content-Type', 'application/json')
        resp.headers['Access-Control-Allow-Origin'] = '*'
        return resp
    except Exception as e:
        return jsonify({'error': str(e)}), 502

@app.route('/', defaults={'path': ''})
@app.route('/<path:path>')
def static_files(path):
    full = os.path.join(DIST_DIR, path)
    if path and os.path.isfile(full):
        return send_from_directory(DIST_DIR, path)
    return send_from_directory(DIST_DIR, 'index.html')

threading.Thread(target=lambda: app.run(port=PORT, host='0.0.0.0', use_reloader=False), daemon=True).start()
import time; time.sleep(2)
print('OK Servidor corriendo en puerto ' + str(PORT))

In [ ]:
# Celda 4: Ngrok tunnel (sin limite de timeout de Cloudflare)
from pyngrok import ngrok, conf

NGROK_TOKEN = '3C4oDG5Pmr1grDKAPfFgppnWLk8_5mSaGQF6TaKrCNXHTCfNf'
conf.get_default().auth_token = NGROK_TOKEN

# Cerrar tunnels anteriores si existen
ngrok.kill()

tunnel = ngrok.connect(PORT, 'http')
tunnel_url = tunnel.public_url

print('\n+------------------------------------------------------+')
print('  OK Abre esta URL en tu browser:')
print('  ' + tunnel_url)
print('+------------------------------------------------------+')
print('  (No cierres esta pestana mientras usas la herramienta)')

In [ ]:
# Celda 5: Keep-alive (corre indefinidamente, es normal)
import time
print('Sesion activa. La herramienta esta disponible en la URL de arriba.')
print('Para terminar: Entorno de ejecucion -> Interrumpir ejecucion')
i = 0
while True:
    time.sleep(60)
    i += 1
    print('[' + str(i) + ' min] activo', end='\r')